# Data Cleaning

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv("./olist_orders_dataset.csv")
items = pd.read_csv("./olist_order_items_dataset.csv")
products = pd.read_csv("./olist_products_dataset.csv")
customers = pd.read_csv("./olist_customers_dataset.csv")
payments = pd.read_csv("./olist_order_payments_dataset.csv")
reviews = pd.read_csv("./olist_order_reviews_dataset.csv")
sellers = pd.read_csv("./olist_sellers_dataset.csv")
geoloc = pd.read_csv("./olist_geolocation_dataset.csv")

In [2]:
# Orders - cast de fechas
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

# Items - cast de fecha
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

# Reviews - cast de fechas
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

In [3]:
orders_clean = orders.copy()
items_clean = items.copy()
products_clean = products.copy()
customers_clean = customers.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
sellers_clean = sellers.copy()
geoloc_clean = geoloc.copy()

### Sellers

In [4]:
# 1. Zero-padding in seller_zip_code_prefix
sellers_clean['seller_zip_code_prefix'] = (sellers_clean['seller_zip_code_prefix'].astype(str).str.zfill(5))

# 2. Normalize seller city
sellers_clean['seller_city'] = (
    sellers_clean['seller_city']
    .str.lower()
    .str.strip()
    )

In [5]:
# Verification zip: all must have 5 characters
assert sellers_clean['seller_zip_code_prefix'].str.len().eq(5).all(), "There are zips with incorrect length"

### Customers

In [6]:
# Zero padding in customer_zip_code_prefix
customers_clean['customer_zip_code_prefix'] = (
    customers_clean['customer_zip_code_prefix']
    .astype(str)
    .str.zfill(5)
)

In [7]:
# Verification zip: all must have 5 characters
assert customers_clean['customer_zip_code_prefix'].str.len().eq(5).all(), "There are zips with incorrect length"

### Payments

In [8]:
# Delete Invalid Payment Records
payments_clean = payments_clean[
    (payments_clean['payment_value'] > 0) &
    (payments_clean['payment_installments'] > 0) &
    (payments_clean['payment_type'] != 'not_defined')
]

In [9]:
assert (payments_clean['payment_value'] <= 0).sum() == 0
assert (payments_clean['payment_installments'] <= 0).sum() == 0
assert (payments_clean['payment_type'] == 'not_defined').sum() == 0

print(f"Removed Records: {len(payments) - len(payments_clean)}")
print(f"Clean Payment Records: {len(payments_clean)}")

Removed Records: 11
Clean Payment Records: 103875


### Reviews

In [10]:
# Drop duplicate review_id, keeping the most recent one
reviews_clean = reviews_clean.sort_values('review_answer_timestamp').drop_duplicates(
    subset='review_id',
    keep='last'
)

In [11]:
assert reviews_clean['review_id'].duplicated().sum() == 0, 'There are still duplicated review_ids'

print(f'Removed Records: {len(reviews) - len(reviews_clean)}')
print(f"Clean Reviews Records: {len(reviews_clean)}")

Removed Records: 814
Clean Reviews Records: 98410


### Order Items

In [12]:
# Flag records with freight_value = 0 as free shipping
items_clean['is_free_shipping'] = items_clean['freight_value'] == 0

In [13]:
print(f"Free shipping records: {items_clean['is_free_shipping'].sum()}")  # should be 383
print(f"Clean Order Items Records: {len(items_clean)}")

Free shipping records: 383
Clean Order Items Records: 112650


### Products

In [14]:
# 1. Rename columns with type
products_clean = products_clean.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

# 2. Fill missing category as 'unknown'
products_clean['product_category_name'] = (
    products_clean['product_category_name'].fillna('unknown')
)

# 3. Flag invalid records (weight or dimensions <= 0)
products_clean['is_physically_invalid'] = (
    (products_clean['product_weight_g'] <= 0) |
    (products_clean['product_length_cm'] <= 0) |
    (products_clean['product_height_cm'] <= 0) |
    (products_clean['product_width_cm'] <= 0)
)

In [15]:
# Check columns were renamed
assert 'product_name_length' in products_clean.columns, 'Column product_name_length does not exists in products'
assert 'product_description_length' in products_clean.columns, 'Column product_description_length does not exists in products'

# Check no more null categories
assert products_clean['product_category_name'].isna().sum() == 0, 'Column cateogry still has null values'

print(f'Unknown category records: {(products_clean['product_category_name'] == 'unknown').sum()}')
print(f'Physically invalid records: {products_clean['is_physically_invalid'].sum()}')
print(f'Clean Products Records: {len(products_clean)}')

Unknown category records: 610
Physically invalid records: 4
Clean Products Records: 32951


### Orders

In [16]:
# Flag records with invalid date sequence
orders_clean['has_invalid_dates'] = (
    (orders_clean['order_purchase_timestamp'] > orders_clean['order_approved_at']) |
    (orders_clean['order_approved_at'] > orders_clean['order_delivered_carrier_date']) |
    (orders_clean['order_delivered_carrier_date'] > orders_clean['order_delivered_customer_date'])
)

In [17]:
print(f"Records with invalid date sequence: {orders_clean['has_invalid_dates'].sum()}")
print(f"Clean Orders Records: {len(orders_clean)}")

Records with invalid date sequence: 1382
Clean Orders Records: 99441


> **Note:** The NB2 profiling reported 1,373 records with invalid date sequences, filtered only on rows 
> where all 4 date columns were non-null. Here the flag is applied to all rows — pandas treats 
> comparisons involving NaT as False, but edge cases where only some dates are null can still trigger 
> the condition, accounting for the 9-record difference.

### Geolocation

In [18]:
# 1. drop exact duplicates
geoloc_clean = geoloc_clean.drop_duplicates()

# 2. filter coordinates outside Brazil
geoloc_clean = geoloc_clean[
    geoloc_clean['geolocation_lat'].between(-34, 5.3) &
    geoloc_clean['geolocation_lng'].between(-74, -28)
]

# 3.Aggregate by zip code prefix (median lat/lng) - one row per prefix
geoloc_agg  = geoloc_clean.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'median'),
    geolocation_lng=('geolocation_lng', 'median'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
).reset_index()

# 4. Zero-padding on zip code prefix
geoloc_agg['geolocation_zip_code_prefix'] = (
    geoloc_agg['geolocation_zip_code_prefix']
    .astype(str)
    .str.zfill(5)
)

In [19]:
exact_duplicates = len(geoloc) - len(geoloc.drop_duplicates())
records_after_dedup = len(geoloc) - exact_duplicates
records_after_filter = len(geoloc_clean)

assert geoloc_agg.duplicated().sum() == 0
assert geoloc_agg['geolocation_zip_code_prefix'].str.len().eq(5).all()

print(f'Removed exact duplicates: {exact_duplicates}')
print(f'Records after data deduplication: {records_after_dedup}')
print(f'Records after coordinate filter: {records_after_filter}')
print(f'Unique zip prefixes after aggregation: {len(geoloc_agg)}')
print(f'Zip prefix length check \n{geoloc_agg['geolocation_zip_code_prefix'].str.len().value_counts()}')

Removed exact duplicates: 261831
Records after data deduplication: 738332
Records after coordinate filter: 738305
Unique zip prefixes after aggregation: 19011
Zip prefix length check 
geolocation_zip_code_prefix
5    19011
Name: count, dtype: int64


#### Export of cleaned datasets

In [20]:
orders_clean.to_csv('./clean_datasets/orders_clean.csv', index=False)
items_clean.to_csv('./clean_datasets/items_clean.csv', index=False)
products_clean.to_csv('./clean_datasets/products_clean.csv', index=False)
customers_clean.to_csv('./clean_datasets/customers_clean.csv', index=False)
payments_clean.to_csv('./clean_datasets/payments_clean.csv', index=False)
reviews_clean.to_csv('./clean_datasets/reviews_clean.csv', index=False)
sellers_clean.to_csv('./clean_datasets/sellers_clean.csv', index=False)
geoloc_agg.to_csv('./clean_datasets/geoloc_agg.csv', index=False)

print("All clean datasets exported successfully.")

All clean datasets exported successfully.
